# SANCR 2014 Cancer Registry Analysis
This notebook performs data loading, cleaning, exploratory analysis, leakage-aware feature selection, classification, hyperparameter tuning, evaluation and export of results using the SANCR 2014 aggregate cancer registry workbook.

**Important:** the target is an aggregate high/lower ASR surveillance label. It is not an individual patient diagnosis or medical-scheme eligibility outcome.


In [ ]:
from pathlib import Path
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import load_workbook
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)
warnings.filterwarnings('ignore')
RANDOM_STATE=42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns',100)
print('Libraries imported successfully.')


In [ ]:
# Exact local file path supplied
DATA_FILE = Path(r"C:\Users\dzauakr\Downloads\Cancer_Registry_Data_2014.xlsx")
if not DATA_FILE.exists():
    raise FileNotFoundError(f"File not found: {DATA_FILE}")
print('Dataset found:', DATA_FILE)

wb_check=load_workbook(DATA_FILE,read_only=True,data_only=True)
print('Worksheets:', wb_check.sheetnames)
required=[f'Page {i:02d}' for i in range(1,11)]
missing=[s for s in required if s not in wb_check.sheetnames]
wb_check.close()
if missing:
    raise ValueError(f'Required worksheets missing: {missing}')


## Data preparation plan
The first ten summary-statistics worksheets are converted into one tidy table. Totals and incomplete lines are excluded. Numeric measures are validated, duplicates are removed and missing ASR values are excluded.


In [ ]:
def parse_summary_sheets(path):
    wb=load_workbook(path,data_only=True)
    records=[]
    for sheet_name in [f'Page {i:02d}' for i in range(1,11)]:
        ws=wb[sheet_name]
        title=' '.join(str(ws.cell(r,1).value or '') for r in [1,2]).upper()
        sex='Female' if 'FEMALE' in title else 'Male' if 'MALE' in title else 'Unknown'
        if 'ALL POPULATION' in title: population='All'
        elif 'ASIAN' in title: population='Asian'
        elif 'BLACK' in title: population='Black'
        elif 'COLOURED' in title: population='Coloured'
        elif 'WHITE' in title: population='White'
        else: population='Unknown'
        for values in ws.iter_rows(min_row=4,values_only=True):
            text=' '.join(str(v) for v in values if v is not None).replace('*','').strip()
            match=re.search(r'(?<![A-Za-z])-?\d+(?:\.\d+)?',text)
            if not match: continue
            site=text[:match.start()].strip()
            nums=re.findall(r'-?\d+(?:\.\d+)?',text[match.start():])[:9]
            if not site or site.lower() in {'total','totals','notes'} or len(nums)<9: continue
            records.append([sheet_name,sex,population,site]+[float(x) for x in nums])
    wb.close()
    cols=['Page','Sex','Population_Group','Site','N_OBS','N_ADJ','Percent','Crude','ASR','LCL','UCL','CumRisk','LifetimeRisk']
    return pd.DataFrame(records,columns=cols)

df=parse_summary_sheets(DATA_FILE).drop_duplicates().dropna(subset=['ASR']).reset_index(drop=True)
if df.empty: raise ValueError('No usable observations were parsed.')
print('Parsed observations:',len(df))
display(df.head(10)); display(df.describe(include='all').T)
print('Missing values:\n',df.isna().sum()); print('Duplicates:',df.duplicated().sum())


## Exploratory Data Analysis
ASR distributions, group differences and correlations are reviewed. The binary modelling target equals 1 when ASR is at or above the median positive ASR and 0 otherwise.


In [ ]:
positive_asr=df.loc[df['ASR']>0,'ASR']
if positive_asr.empty: raise ValueError('No positive ASR values found.')
median_asr=positive_asr.median()
df['High_ASR']=(df['ASR']>=median_asr).astype(int)
print('Median positive ASR threshold:',median_asr)
display(df['High_ASR'].value_counts().sort_index().to_frame('Count'))
fig,axes=plt.subplots(1,2,figsize=(14,5))
sns.histplot(data=df,x='ASR',bins=30,kde=True,ax=axes[0]); axes[0].axvline(median_asr,color='red',ls='--'); axes[0].set_title('ASR distribution')
sns.boxplot(data=df,x='Sex',y='ASR',hue='Population_Group',ax=axes[1]); axes[1].set_title('ASR by summary group')
plt.tight_layout(); plt.show()

numeric=['N_OBS','N_ADJ','Percent','Crude','ASR','LCL','UCL','CumRisk','LifetimeRisk']
plt.figure(figsize=(11,8)); sns.heatmap(df[numeric].corr(),annot=True,fmt='.2f',cmap='coolwarm',center=0); plt.title('Correlation matrix'); plt.tight_layout(); plt.show()


## Feature selection and leakage control
Because `High_ASR` is derived directly from ASR, ASR itself is excluded. LCL, UCL, cumulative risk and lifetime risk are also excluded because they are closely tied to ASR. Logistic Regression applies ANOVA F selection inside cross-validation; tree models perform embedded nonlinear selection.


In [ ]:
features=['Sex','Population_Group','N_OBS','N_ADJ','Percent','Crude']
X=df[features].copy(); y=df['High_ASR'].copy()
if len(X)<20 or y.nunique()!=2 or y.value_counts().min()<5:
    raise ValueError('Insufficient observations/classes for modelling.')
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=RANDOM_STATE)
num=['N_OBS','N_ADJ','Percent','Crude']; cat=['Sex','Population_Group']
pre=ColumnTransformer([
 ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
 ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)
])
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
print('Training:',len(X_train),'Testing:',len(X_test))


## Model training and retraining
Three classifiers are compared. Grid search functions as controlled retraining across regularisation, selected-feature count, class weighting, tree depth, minimum leaf size, estimator count and learning rate. Cross-validated F1 is used for model selection.


In [ ]:
models={
 'Logistic Regression':(Pipeline([('pre',pre),('select',SelectKBest(f_classif)),('model',LogisticRegression(max_iter=3000,random_state=RANDOM_STATE))]),{'select__k':[4,6,8],'model__C':[.01,.1,1,10],'model__class_weight':[None,'balanced']}),
 'Random Forest':(Pipeline([('pre',pre),('model',RandomForestClassifier(random_state=RANDOM_STATE))]),{'model__n_estimators':[150,300],'model__max_depth':[None,5,10],'model__min_samples_leaf':[1,2,4],'model__class_weight':[None,'balanced']}),
 'Gradient Boosting':(Pipeline([('pre',pre),('model',GradientBoostingClassifier(random_state=RANDOM_STATE))]),{'model__n_estimators':[50,100,200],'model__learning_rate':[.03,.05,.1],'model__max_depth':[1,2,3]})
}
rows=[]; searches={}
for name,(model,grid) in models.items():
    search=GridSearchCV(model,grid,cv=cv,scoring='f1',n_jobs=-1,refit=True)
    search.fit(X_train,y_train)
    pred=search.predict(X_test); prob=search.predict_proba(X_test)[:,1]
    searches[name]=search
    rows.append({'Model':name,'CV_F1':search.best_score_,'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0),'ROC_AUC':roc_auc_score(y_test,prob),'Best_Params':search.best_params_})
results=pd.DataFrame(rows).sort_values('CV_F1',ascending=False).reset_index(drop=True)
display(results)


## Evaluation
Accuracy reports overall correctness. Precision measures the reliability of a high-ASR prediction, recall measures capture of high-ASR observations, F1 balances precision and recall, and ROC-AUC evaluates probability ranking.


In [ ]:
best_name=results.loc[0,'Model']; best_model=searches[best_name]
pred=best_model.predict(X_test); prob=best_model.predict_proba(X_test)[:,1]
print('Selected model:',best_name); print('Best parameters:',best_model.best_params_)
print(classification_report(y_test,pred,target_names=['Lower ASR','High ASR'],zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['Lower ASR','High ASR'],cmap='Blues'); plt.show()
RocCurveDisplay.from_predictions(y_test,prob,name=best_name); plt.show()


## Export and limitation
Outputs are saved in the same Downloads folder as the workbook. The records are aggregated cancer-site/group observations, not patient-level cases. Results therefore support an academic surveillance proof of concept only.


In [ ]:
OUTPUT_FOLDER=DATA_FILE.parent
df.to_csv(OUTPUT_FOLDER/'SANCR_2014_Cleaned_Data.csv',index=False)
results.to_csv(OUTPUT_FOLDER/'SANCR_2014_Model_Results.csv',index=False)
test_results=X_test.copy(); test_results['Actual']=y_test; test_results['Predicted']=pred; test_results['Probability_High_ASR']=prob
test_results.to_csv(OUTPUT_FOLDER/'SANCR_2014_Test_Predictions.csv',index=False)
print('Outputs saved to:',OUTPUT_FOLDER)
